# RI-JK UHF Hessian：CP-HF 分解 (2) 响应函数与方程求解

本文档对应 `02-5-decomp_cphf_2.ipynb` 的 UHF 版本。重点：

1. UHF 的 response function `vresp` 接受 `[2, ..., nao, nao]` 的密度扰动，返回 $V^\sigma = J[D^\alpha + D^\beta] - K^\sigma[D^\sigma]$。
2. `vind` 把 `mo1` 转回 AO 空间得到 dm1，调用 `vresp`，再投回 MO 空间。两个自旋通道在 `vind` 中**耦合**（J 部分共享，K 部分各自分离）。
3. 由于 `nocc_α ≠ nocc_β`，`vind` 接受/返回的格式是 `[nset, nmoa*nocca + nmob*noccb]` 的**展平向量**，而**不是** 5-dim 张量。
4. 手写 CP-HF 求解流程，复现 PySCF 的 `ucphf.solve_withs1`。

In [1]:
from pyscf import gto, scf, lib, df, hessian
from pyscf.hessian import uhf as uhf_hess
from pyscf.scf import ucphf
import numpy as np
from functools import partial
from pyscf.df.grad.rhf import _int3c_wrapper

lib.num_threads(16)
np.set_printoptions(5, suppress=True, linewidth=150)
np.einsum = partial(np.einsum, optimize="greedy")

In [2]:
xyz = """
N  0   0   0
H  1.0 0.1 0.2
H  0.3 1.1 0.2
H  0.1 0.1 1.2
"""

mol = gto.Mole(atom=xyz, basis="def2-TZVP", charge=2, spin=2, max_memory=32000).build()

In [3]:
mf = scf.UHF(mol).density_fit()
mf.mo_coeff = np.load("nh3_u_hf.npz")["mo_coeff"]
mf.mo_occ = np.load("nh3_u_hf.npz")["mo_occ"]
mf.mo_energy = np.load("nh3_u_hf.npz")["mo_energy"]
mf.with_df.build()
mf.converged = True

In [4]:
mf_hess = mf.Hessian().run()
de_ref = mf_hess.de.copy()
print("de_ref shape:", de_ref.shape)

de_ref shape: (4, 4, 3, 3)


In [5]:
de_cphf = np.load("nh3_u_hf_decomp.npz")["de_cphf"]

In [6]:
α, β = 0, 1

mo_coeff = mf.mo_coeff
mo_occ = mf.mo_occ
mo_energy = mf.mo_energy
nao = mo_coeff.shape[1]
nmo = mo_coeff.shape[2]

mocc = [mo_coeff[x][:, mo_occ[x] > 0] for x in (α, β)]
mvir = [mo_coeff[x][:, mo_occ[x] == 0] for x in (α, β)]
nocc = [mocc[x].shape[1] for x in (α, β)]
nvir = [mvir[x].shape[1] for x in (α, β)]
eocc = [mo_energy[x][mo_occ[x] > 0] for x in (α, β)]
evir = [mo_energy[x][mo_occ[x] == 0] for x in (α, β)]

dm0 = np.zeros((2, nao, nao))
dm0[α] = mocc[α] @ mocc[α].T
dm0[β] = mocc[β] @ mocc[β].T
dm0_s = dm0.sum(axis=0)

natm = mol.natm
atmlst = range(natm)
aoslices = mol.aoslice_by_atom()
aux = mf.with_df.auxmol
auxslices = aux.aoslice_by_atom()
naux = aux.nao

print("nocc:", nocc, " nvir:", nvir)

nocc: [5, 3]  nvir: [44, 46]


In [7]:
def ovlp_deriv1_generator(mol):
    int1e_ipovlp = mol.intor("int1e_ipovlp")

    def get_ovlp_deriv_at_atoms(A):
        shl0, shl1, p0, p1 = aoslices[A]
        s1ao = np.zeros((3, nao, nao))
        s1ao[:, p0:p1, :] += - int1e_ipovlp[:, p0:p1]
        s1ao[:, :, p0:p1] += - int1e_ipovlp[:, p0:p1].transpose(0, 2, 1)
        return s1ao
    return get_ovlp_deriv_at_atoms

In [8]:
f1ao = mf_hess.make_h1(mo_coeff, mo_occ)
f1ao = [np.asarray(f1ao[x]) for x in (α, β)]
# mf_hess.solve_mo1 返回的 mo1 已是 bra-transformed C @ U（即 mo1_bra）
mo1_bra, mo_e1 = mf_hess.solve_mo1(mo_energy, mo_coeff, mo_occ, f1ao)
mo1_bra = [np.asarray(mo1_bra[x]) for x in (α, β)]
mo_e1 = [np.asarray(mo_e1[x]) for x in (α, β)]

In [9]:
int2c2e = aux.intor("int2c2e")
int2c2e_inv = np.linalg.inv(int2c2e)
int2c2e_ip1 = aux.intor("int2c2e_ip1")
int3c2e = _int3c_wrapper(mol, aux, "int3c2e", "s1")()
int3c2e_ip1 = _int3c_wrapper(mol, aux, "int3c2e_ip1", "s1")().reshape([3, nao, nao, naux])
int3c2e_ip2 = _int3c_wrapper(mol, aux, "int3c2e_ip2", "s1")().reshape([3, nao, nao, naux])

## Response Function

### 原始 vresp

UHF 的 `vresp` 接收 `dm1` 形状为 `[2, nao, nao]` 或 `[2, nset, nao, nao]`，返回相同形状的响应。对每个自旋通道 $\sigma$：

$$
V^\sigma[\delta D] = J[\delta D^\alpha + \delta D^\beta] - K^\sigma[\delta D^\sigma]
$$

In [10]:
vresp = mf.gen_response()

In [11]:
dm_rand = np.random.rand(2, nao, nao)
dm_rand += dm_rand.transpose(0, 2, 1)

# vresp 与直接 get_veff 应一致（这里 get_veff 返回总势的两个自旋分量）
veff_ref = mf.get_veff(dm=dm_rand)
print("vresp matches get_veff(dm_diff):", np.allclose(vresp(dm_rand), veff_ref - mf.get_veff(dm=np.zeros((2, nao, nao)))))

vresp matches get_veff(dm_diff): True


In [12]:
# 显式 einsum 重写：J 部分用总密度，K 部分按自旋
dm_rand_s = dm_rand.sum(axis=0)
v_explicit = np.zeros_like(dm_rand)
j_part = np.einsum("uvP, PQ, klQ, kl -> uv", int3c2e, int2c2e_inv, int3c2e, dm_rand_s)
for x in (α, β):
    k_part = np.einsum("uvP, PQ, klQ, vl -> uk", int3c2e, int2c2e_inv, int3c2e, dm_rand[x])
    v_explicit[x] = j_part - k_part

print("explicit einsum matches vresp:", np.allclose(v_explicit, vresp(dm_rand)))

explicit einsum matches vresp: True


### 原始 vind（PySCF）

PySCF 的 `gen_vind` 把 `mo1` 拼成一个 1D 向量 `[nmoa*nocca + nmob*noccb]`，里面顺序是 α 在前、β 在后。我们先用 PySCF 版本生成随机扰动并与显式 einsum 公式比较。

In [13]:
# 随机扰动：分别为两个自旋通道生成 mo1，形状不同
mo1_rand = [
    np.random.rand(3, nmo, nocc[α]),
    np.random.rand(3, nmo, nocc[β]),
]

# 拼成 PySCF 所需的展平形式 [nset=3, nmoa*nocca + nmob*noccb]
def pack_mo1(mo1_list):
    nset = mo1_list[α].shape[0]
    return np.hstack([mo1_list[α].reshape(nset, -1), mo1_list[β].reshape(nset, -1)])

def unpack_mo1(flat):
    nset = flat.shape[0]
    n_a = nmo * nocc[α]
    a = flat[:, :n_a].reshape(nset, nmo, nocc[α])
    b = flat[:, n_a:].reshape(nset, nmo, nocc[β])
    return [a, b]

mo1_rand_flat = pack_mo1(mo1_rand)
print("packed shape:", mo1_rand_flat.shape, "= [nset, nmoa*nocca + nmob*noccb] =", f"[3, {nmo*nocc[α]}+{nmo*nocc[β]}={nmo*nocc[α]+nmo*nocc[β]}]")

packed shape: (3, 392) = [nset, nmoa*nocca + nmob*noccb] = [3, 245+147=392]


In [14]:
# PySCF 内部 fvind
fvind = uhf_hess.gen_vind(mf, mo_coeff, mo_occ)
v_pyscf_flat = fvind(mo1_rand_flat)
v_pyscf = unpack_mo1(v_pyscf_flat)

# 显式重写：先 mo -> ao 得 dm1（每个自旋单独构造），调用 vresp，再投回 MO 空间
dm1_rand = np.zeros((2, 3, nao, nao))
for x in (α, β):
    # dm1 = C @ mo1 @ Cocc.T  + symmetrization；注意 UHF 无 RHF 的 *2 因子
    dm_half = mo_coeff[x] @ mo1_rand[x] @ mocc[x].T
    dm1_rand[x] = dm_half + dm_half.transpose(0, 2, 1)

# vresp 接受 [2, nset, nao, nao]
v1ao = vresp(dm1_rand)
v_manual = [mo_coeff[x].T @ v1ao[x] @ mocc[x] for x in (α, β)]

for x in (α, β):
    print(f"spin {x}: match PySCF fvind:", np.allclose(v_manual[x], v_pyscf[x]))

spin 0: match PySCF fvind: True
spin 1: match PySCF fvind: True


### vind 显式分解（J + K，更利于优化）

把 vresp 进一步展开到 einsum 层级，并且把 occupied 指标提前缩并到 mocc 上。对每个 $\mathbb{A}$ 扰动 $U^{\mathbb{A}, \sigma}$：

$$
V^{\mathbb{A}, \sigma}_{p i} = \sum_\tau 2 J^{\mathbb{A}, \tau}_{p i}[U^{\mathbb{A}, \tau}] - K^{\mathbb{A}, \sigma}_{p i}[U^{\mathbb{A}, \sigma}]
$$

因子说明：
- J 部分对 $\tau$ 求和（$\alpha + \beta$）。这里的因子 `2` 来自 `mo1` + `mo1^T` 对称化（dm1_rand 显式写 `dm + dm.T`）。
- K 部分只与同自旋耦合，因子 `1`（同样来自对称化，但 K 只半边贡献）。

In [15]:
# 完整显式分解：直接用 mo1_rand（未做半变换）
v_decomp = [np.zeros_like(mo1_rand[x]) for x in (α, β)]

for x in (α, β):
    # J: 两个自旋通道都贡献
    for τ in (α, β):
        v_decomp[x] += 2 * np.einsum(
            "uvP, PQ, klQ, Aqj, kq, lj, up, vi -> Api",
            int3c2e, int2c2e_inv, int3c2e, mo1_rand[τ], mo_coeff[τ], mocc[τ], mo_coeff[x], mocc[x],
        )
    # K: 仅同自旋
    v_decomp[x] -= 1 * np.einsum(
        "uvP, PQ, klQ, Aqj, vq, lj, up, ki -> Api",
        int3c2e, int2c2e_inv, int3c2e, mo1_rand[x], mo_coeff[x], mocc[x], mo_coeff[x], mocc[x],
    )
    v_decomp[x] -= 1 * np.einsum(
        "uvP, PQ, klQ, Aqj, lq, vj, up, ki -> Api",
        int3c2e, int2c2e_inv, int3c2e, mo1_rand[x], mo_coeff[x], mocc[x], mo_coeff[x], mocc[x],
    )

for x in (α, β):
    print(f"spin {x}: match PySCF fvind:", np.allclose(v_decomp[x], v_pyscf[x]))

spin 0: match PySCF fvind: True
spin 1: match PySCF fvind: True


In [16]:
# Half-transformed 形式：先把 mo1 的左 AO 指标变换到 AO 空间，再做主缩并
mo1_half = [mo_coeff[x] @ mo1_rand[x] for x in (α, β)]  # shape [3, nao, nocc_σ]

v_half = [np.zeros((3, nao, nocc[x])) for x in (α, β)]
for x in (α, β):
    for τ in (α, β):
        v_half[x] += 2 * np.einsum(
            "uvP, PQ, klQ, Akj, lj, vi -> Aui",
            int3c2e, int2c2e_inv, int3c2e, mo1_half[τ], mocc[τ], mocc[x],
        )
    v_half[x] -= 1 * np.einsum(
        "uvP, PQ, klQ, Avj, lj, ki -> Aui",
        int3c2e, int2c2e_inv, int3c2e, mo1_half[x], mocc[x], mocc[x],
    )
    v_half[x] -= 1 * np.einsum(
        "uvP, PQ, klQ, Akj, vj, li -> Aui",
        int3c2e, int2c2e_inv, int3c2e, mo1_half[x], mocc[x], mocc[x],
    )

# 投回 MO 空间
v_half_mo = [np.einsum("Aui, up -> Api", v_half[x], mo_coeff[x]) for x in (α, β)]

for x in (α, β):
    print(f"spin {x}: half-trans matches PySCF fvind:", np.allclose(v_half_mo[x], v_pyscf[x]))

spin 0: half-trans matches PySCF fvind: True
spin 1: half-trans matches PySCF fvind: True


## CP-HF recover（手写求解）

把 `f1ao`、`s1ao` 准备成 `[natm*3, nao, nao]` 后调用一遍 Krylov 求解 + 收尾，复现 PySCF 的 `ucphf.solve_withs1`。两个自旋通道在 Krylov 迭代中通过 `vind` 耦合。

In [17]:
s1ao_full = np.array([ovlp_deriv1_generator(mol)(A) for A in range(natm)])  # [natm, 3, nao, nao]
s1ao_flat = s1ao_full.reshape(-1, nao, nao)  # [natm*3, nao, nao]
f1ao_flat = [f1ao[x].reshape(-1, nao, nao) for x in (α, β)]

In [18]:
level_shift = 0
e_ai = [1.0 / (evir[x][:, None] - eocc[x][None, :] + level_shift) for x in (α, β)]

In [19]:
f1mo = [mo_coeff[x].T @ f1ao_flat[x] @ mocc[x] for x in (α, β)]
s1mo = [mo_coeff[x].T @ s1ao_flat @ mocc[x] for x in (α, β)]
hsmo = [f1mo[x] - s1mo[x] * eocc[x] for x in (α, β)]

# CP-HF base 初值：vir 块用 -hsmo / (ea - ei)，occ 块用 -0.5 * s1mo
mo1_base = [np.zeros_like(hsmo[x]) for x in (α, β)]
for x in (α, β):
    mo1_base[x][:, nocc[x]:] = -hsmo[x][:, nocc[x]:] * e_ai[x]
    mo1_base[x][:, :nocc[x]] = -s1mo[x][:, :nocc[x]] * 0.5

# 拼成 1D 展平向量；pack/unpack 自动按当前批维度处理
n_a = nmo * nocc[α]
n_b = nmo * nocc[β]
nset = natm * 3

def pack_uhf(arr_list):
    n = arr_list[α].shape[0]
    return np.hstack([arr_list[α].reshape(n, -1), arr_list[β].reshape(n, -1)])

def unpack_uhf(flat):
    n = flat.shape[0]
    a = flat[:, :n_a].reshape(n, nmo, nocc[α])
    b = flat[:, n_a:].reshape(n, nmo, nocc[β])
    return [a, b]

mo1_base_flat = pack_uhf(mo1_base)

fvind_pyscf = uhf_hess.gen_vind(mf, mo_coeff, mo_occ)

def vind_vo(mo1_flat):
    mo1_flat = mo1_flat.reshape(-1, n_a + n_b)
    v_flat = fvind_pyscf(mo1_flat).reshape(-1, n_a + n_b)
    if level_shift != 0:
        v_flat -= mo1_flat * level_shift
    v_list = unpack_uhf(v_flat)
    for x in (α, β):
        v_list[x][:, nocc[x]:, :] *= e_ai[x]
        v_list[x][:, :nocc[x], :] = 0
    return pack_uhf(v_list)

mo1_sol_flat = lib.krylov(vind_vo, mo1_base_flat)
mo1_sol = unpack_uhf(mo1_sol_flat)

# occupied 块强制覆盖回 -0.5 * s1mo
for x in (α, β):
    mo1_sol[x][:, :nocc[x]] = mo1_base[x][:, :nocc[x]]

# 最后一步：根据完整 vind 收回 vir 块和 occ-occ 块的 mo_e1
v_final = unpack_uhf(fvind_pyscf(pack_uhf(mo1_sol)))
mo_e1_sol = [None, None]
for x in (α, β):
    hsmo[x] = f1mo[x] - s1mo[x] * eocc[x] + v_final[x]
    mo1_sol[x][:, nocc[x]:] = hsmo[x][:, nocc[x]:] / (eocc[x] - evir[x][:, None])
    mo_e1_sol[x] = hsmo[x][:, :nocc[x]] + mo1_sol[x][:, :nocc[x]] * (eocc[x][:, None] - eocc[x])

# 手写求解得到的是未 bra-transform 的 U_{p i}^{A, σ}，按 [natm, 3, nmo, nocc_σ] 整理
mo1 = [mo1_sol[x].reshape(natm, 3, nmo, nocc[x]) for x in (α, β)]
mo_e1_sol = [mo_e1_sol[x].reshape(natm, 3, nocc[x], nocc[x]) for x in (α, β)]

# 与 PySCF mf_hess.solve_mo1 返回的 mo1_bra 对比（后者已经过 bra-transform）
for x in (α, β):
    mo1_btr = np.einsum("pq, Atqi -> Atpi", mo_coeff[x], mo1[x])
    print(f"spin {x}: manual mo1 (after bra-transform) matches PySCF mo1_bra:", np.allclose(mo1_btr, mo1_bra[x]))
    print(f"spin {x}: manual mo_e1 matches PySCF:", np.allclose(mo_e1_sol[x], mo_e1[x]))

spin 0: manual mo1 (after bra-transform) matches PySCF mo1_bra: True
spin 0: manual mo_e1 matches PySCF: True
spin 1: manual mo1 (after bra-transform) matches PySCF mo1_bra: True
spin 1: manual mo_e1 matches PySCF: True


## 存储未 bra-transform 的 `mo1`

将手写求解得到的 $U_{p i}^{\mathbb{A}, \sigma}$（未 bra-transform）追加保存到 npz，用作后续 pyhessref 模块的测试基准。`05-4` 中已经保存了 bra-transformed 版本 `mo1_bra_a`/`mo1_bra_b`。

## 小结

- UHF 的 `vresp` / `vind` 在两个自旋通道之间通过 J（共享总密度）耦合，K 仅同自旋耦合。
- 数据布局：`mo1` / `mo_e1` 因 `nocc_α ≠ nocc_β` 不能用 5-dim 张量，需用 `[α 部分, β 部分]` 列表 / 拼接为 `[nset, n_a + n_b]` 展平向量。
- 手写 CP-HF 求解（基于 PySCF 的 `fvind`）与 `mf_hess.solve_mo1` 结果一致。
- 命名约定：
  - `mo1_a` / `mo1_b`（未 bra-transform）：$U_{p i}^{\mathbb{A}, \sigma}$，shape `[natm, 3, nmo, nocc_σ]`
  - `mo1_bra_a` / `mo1_bra_b`（bra-transform）：$U_{\mu i}^{\mathbb{A}, \sigma} = \sum_p C_{\mu p}^\sigma U_{p i}^{\mathbb{A}, \sigma}$，shape 相同
- 下一步（`05-6-decomp_cphf_3.ipynb`）：将外部 Krylov 求解器（`krylov_block`）应用到 UHF CP-HF，并核验等价性。

In [20]:
dat = dict(np.load("nh3_u_hf_decomp.npz"))
dat.update({
    "mo1_a": mo1[α],
    "mo1_b": mo1[β],
})
np.savez("nh3_u_hf_decomp.npz", **dat)
print("Saved mo1_a/mo1_b (un-bra-transformed) with shapes:", mo1[α].shape, mo1[β].shape)

Saved mo1_a/mo1_b (un-bra-transformed) with shapes: (4, 3, 49, 5) (4, 3, 49, 3)
